<a href="https://colab.research.google.com/github/vaishnavip213/SmartCrop/blob/main/Crop_recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Train_Dataset.csv to Train_Dataset.csv


In [3]:
import pandas as pd
df = pd.read_csv("Train_Dataset.csv")

In [4]:
df = df.drop(columns=['Unnamed: 0'])

In [5]:
print(df.shape)

(18079, 7)


In [6]:
df.head()

,Crop,N,P,K,pH,rainfall,temperature
0,barley,70,40,45,5.54,75.32,22.676000
1,sunflower,50,60,30,5.54,297.66,29.566667
2,sweetpotato,90,20,120,5.02,689.88,29.037273
3,rice,80,40,40,5.66,297.66,29.566667
4,soyabean,20,60,20,5.38,1011.49,30.430000


In [7]:
print("Number of crops:", df['Crop'].nunique())
print(df['Crop'].unique())
print()
print(df['Crop'].value_counts())

Number of crops: 40
['barley' 'sunflower' 'sweetpotato' 'rice' 'soyabean' 'jowar' 'potato'
 'turmeric' 'jute' 'maize' 'moong' 'onion' 'cabbage' 'ragi' 'banana'
 'rapeseed' 'wheat' 'horsegram' 'coriander' 'mango' 'tomato' 'cotton'
 'brinjal' 'garlic' 'blackgram' 'cardamom' 'blackpepper' 'orange'
 'bittergourd' 'papaya' 'grapes' 'ladyfinger' 'pineapple' 'bottlegourd'
 'jackfruit' 'pumpkin' 'cauliflower' 'cucumber' 'drumstick' 'radish']

Crop
rice           2245
maize          1976
moong          1327
wheat          1287
rapeseed       1118
potato         1083
jowar          1055
onion          1047
sunflower       737
barley          654
cotton          613
sweetpotato     576
ragi            557
horsegram       545
turmeric        528
banana          507
coriander       473
soyabean        442
garlic          415
jute            211
blackpepper     119
mango            66
tomato           59
papaya           56
brinjal          55
cardamom         54
ladyfinger       41
orange          

In [8]:
df.isnull().sum()   # should show 0 for every column

,0
Crop,0
N,0
P,0
K,0
pH,0
rainfall,0
temperature,0


In [9]:
from sklearn.model_selection import train_test_split

X = df[['N', 'P', 'K', 'pH', 'rainfall', 'temperature']]
y = df['Crop']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # <-- important: this ensures even rare crops (like pumpkin, 9 rows) appear in both train and test in proportion
)
print(X_train.shape, X_test.shape)

(14463, 6) (3616, 6)


In [10]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'   # <-- tells the model to pay more attention to rare crops instead of ignoring them
)
model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

In [11]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.9269911504424779
              precision    recall  f1-score   support

      banana       1.00      1.00      1.00       101
      barley       1.00      1.00      1.00       131
 bittergourd       1.00      1.00      1.00         2
   blackgram       1.00      1.00      1.00         2
 blackpepper       1.00      1.00      1.00        24
 bottlegourd       1.00      1.00      1.00         3
     brinjal       1.00      1.00      1.00        11
     cabbage       1.00      1.00      1.00         5
    cardamom       1.00      1.00      1.00        11
 cauliflower       1.00      1.00      1.00         5
   coriander       1.00      1.00      1.00        95
      cotton       1.00      1.00      1.00       123
    cucumber       1.00      1.00      1.00         4
   drumstick       1.00      1.00      1.00         3
      garlic       1.00      1.00      1.00        83
      grapes       1.00      1.00      1.00         4
   horsegram       1.00      1.00      1.00       10

In [12]:
sample = [[60, 30, 30, 6.5, 100, 23]]  # N, P, K, pH, rainfall, temperature — roughly wheat-like conditions
prediction = model.predict(sample)
print(prediction)

['wheat']


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [13]:
import joblib
joblib.dump(model, "crop_model.pkl")

['crop_model.pkl']